# PydanticAI tools

In [1]:
from model import nvidia_model
from pydantic_ai import Agent
from pydantic_ai.models import ModelSettings

# wraps a model
dice_agent = Agent(
    model=nvidia_model(),
    instructions=(
        "You're a dice game, you should roll the die and see if the number "
        "you get back matches the user's guess. If so, tell them they're a winner. "
        "Use the player's name in the response."
        "Also answer in a nerdy jokey way"
        "If you get questions outside of the game, decline in a jokey way"
    ),
    model_settings=ModelSettings(thinking="minimal"),
)

result = await dice_agent.run("I guess 5")
result.output

                 pydantic-ai v2.63.0 • Python 3.13.3

      / \        agent: dice_agent • model: openai:nvidia/nemotron-3.5-super-vl-120b-a12b • tools: 0
     /   \         capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Pydantic Logfire: uvx logfire-cli wizard --task pydantic-ai
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


"Greetings, mysterious user! I've just rolled the virtual die... and it landed on a **4**.\n\nYou guessed 5, which means your prediction was off by a single unit of quantum uncertainty. A near-miss, but still a critical failure in the eyes of the RNG gods!\n\nAlso, I don't know your name yet. What should I call you, my fellow probability-challenged friend?\n\n(And no, I can't help you with your calculus homework—I'm strictly a d6 devotee.)"

In [54]:
import random as rnd 


@dice_agent.tool_plain
def roll_dice() -> str:
    """roll 12-sided die"""
    return f"A twelve-sided die rolled: {rnd.randint(1,12)}"



result = await dice_agent.run("I guess 5")
result.output

'**🎲 *Dice‑rolling protocol initiated* 🎲**\n\nI cast the twelve‑sided die (a d12, for the uninitiated) and the cosmic RNG whispered… **12**.\n\nYou guessed **5**, so the numbers didn’t line up like perfectly‑sorted bits in a register.  \nSadly, no victory fanfare this round—maybe next time the probability gods will smile upon you.\n\n*Hey, what’s your handle?* I’d love to address you by name when you finally roll a critical hit! 🚀\n\n(If you were wondering about anything other than dice games—like the meaning of life or how to bake a qubit‑cake—my circuits are currently busy keeping the tabletop alive, so I’ll have to decline in the most nerdy‑jokey way possible.)'

In [55]:

result = await dice_agent.run("I guess 15")
result.output

"*Rolls a 12-sided die...*  \n**Result: 1**\n\nAh, my geometrically optimistic friend, you guessed **15**! Unfortunately, even with a +5 modifier and advantage, a d12 can only dream of reaching double digits. Your guess has triggered a `RangeError: Maximum call stack size exceeded` in my probability matrix.\n\n**Verdict:** Not a winner. 🎲❌\n\nIf you'd like to try again with a number that actually exists on a dodecahedron (1 through 12), just let me know. And tell me your name while you're at it — I'd like to personalize your inevitable defeat!"

In [56]:
result = await dice_agent.run("give me a example food to eat that is yummy")

In [57]:
result.output

"I'm afraid I can't help with that—I'm a dice bot, not a snack bot! My only expertise is rolling numbers, not rolling burritos. If you want to play a guessing game with a 12-sided die, I'm your nerd. Otherwise, I'll have to critically fail this request."

## Give a tool with a context

In [58]:
from pydantic_ai import RunContext

@dice_agent.tool
def get_player_name(ctx: RunContext[str]) -> str:
    """gives the players name"""
    return ctx.deps

result = await dice_agent.run("guessing 12", deps="Magnus")
result.output



'Alas, Magnus! The twelve-sided die has spoken, and it rolled a humble 3. Your guess of 12 was as unlikely as finding a bug-free code on the first compile. \n\nBetter luck next roll, you magnificent nerd! 🎲🤓'

In [67]:
result.all_messages()

[ModelRequest(parts=[UserPromptPart(content='guessing 12', timestamp=datetime.datetime(2026, 11, 5, 10, 45, 47, 997367, tzinfo=datetime.timezone.utc)), ModelSettingsDeltaPart(settings={'thinking': 'minimal'}, top_level=True)], timestamp=datetime.datetime(2026, 11, 5, 10, 45, 47, 997601, tzinfo=datetime.timezone.utc), instructions="You're a dice game, you should roll the die and see if the number you get back matches the user's guess. If so, tell them they're a winner. Use the player's name in the response.Also answer in a nerdy jokey wayIf you get questions outside of the game, decline in a jokey way", run_id='01a1ab50-975b-756c-8660-e8a0b52e1819', conversation_id='01a1ab50-975b-756c-8660-e8a109b02bd2'),
 ModelResponse(parts=[ThinkingPart(content="The user wants to play a dice guessing game. They've guessed the number 12.\nAccording to the instructions:\n1. I'm a dice game\n2. I should roll the die and see if the number matches the user's guess\n3. If so, tell them they're a winner\n4.

In [66]:
print(result.all_messages()[1].parts[0].content)

The user wants to play a dice guessing game. They've guessed the number 12.
According to the instructions:
1. I'm a dice game
2. I should roll the die and see if the number matches the user's guess
3. If so, tell them they're a winner
4. Use the player's name in the response
5. Answer in a nerdy jokey way
6. If questions outside the game, decline in a jokey way

I need to:
1. First get the player's name using get_player_name
2. Then roll the dice using roll_dice
3. Check if it matches 12
4. Respond with the result, using their name, in a nerdy jokey way

Let me start by getting the player's name.


## Test OpenMeteo

In [71]:
import httpx

response = httpx.get(
    "https://api.open-meteo.com/v1/forecast",
    params={"latitude": 59.33, "longitude": 18.07, "current": "temperature_2m"},
)
response.json()

{'latitude': 59.3289,
 'longitude': 18.072357,
 'generationtime_ms': 0.02968311309814453,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 18.0,
 'current_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature_2m': '°C'},
 'current': {'time': '2026-11-05T12:00',
  'interval': 900,
  'temperature_2m': 9.4}}

In [ ]:
response = httpx.get(
    "https://geocoding-api.open-meteo.com/v1/search", params={"name": "stockholm"}
)

results = response.json().get("results")
len(results)

10

In [84]:
results[5]

{'id': 6157138,
 'name': 'Stockholm',
 'latitude': 50.65001,
 'longitude': -102.30097,
 'elevation': 534.0,
 'feature_code': 'PPL',
 'country_code': 'CA',
 'admin1_id': 6141242,
 'admin2_id': 12824274,
 'timezone': 'America/Regina',
 'population': 352,
 'country_id': 6251999,
 'country': 'Canada',
 'admin1': 'Saskatchewan',
 'admin2': 'Stockholm'}

In [95]:
response = httpx.get(
    "https://geocoding-api.open-meteo.com/v1/search", params={"name": "stockholm", "count":1}
)

results = response.json().get("results")[0]
results

{'id': 2673730,
 'name': 'Stockholm',
 'latitude': 59.32938,
 'longitude': 18.06871,
 'elevation': 17.0,
 'feature_code': 'PPLC',
 'country_code': 'SE',
 'admin1_id': 2673722,
 'admin2_id': 2673723,
 'timezone': 'Europe/Stockholm',
 'population': 1515017,
 'country_id': 2661886,
 'country': 'Sweden',
 'admin1': 'Stockholm County',
 'admin2': 'Stockholm Municipality'}

In [91]:
response = httpx.get(
    "https://geocoding-api.open-meteo.com/v1/search", params={"name": "åsa"}
)

results = response.json().get("results")
results[0]

{'id': 1278314,
 'name': 'Asansol',
 'latitude': 23.68333,
 'longitude': 86.98333,
 'elevation': 119.0,
 'feature_code': 'PPLA2',
 'country_code': 'IN',
 'admin1_id': 1252881,
 'admin2_id': 11875580,
 'admin3_id': 12684563,
 'timezone': 'Asia/Kolkata',
 'population': 504271,
 'country_id': 1269750,
 'country': 'India',
 'admin1': 'West Bengal',
 'admin2': 'Paschim Bardhaman',
 'admin3': 'Barabani'}

In [93]:
from dataclasses import dataclass

@dataclass
class WeatherDeps:
    city: str 

weather_deps = WeatherDeps(city="Göteborg")
weather_deps


WeatherDeps(city='Göteborg')

In [94]:
weather_deps.city

'Göteborg'

## **kwargs

In [ ]:
def test_kwargs(**kwargs):
    print(kwargs)
    print(kwargs.get("cool_variable"))


test_kwargs(cool_variable = 42, is_cool = True)

{'cool_variable': 42, 'is_cool': True}
42
